In [ ]:
def get_soc_and_diffusivities_sobol(
    soc_levels,
    num_samples_per_soc,
    log_lower=-14,
    log_upper=-12,
):
    sampler_log = qmc.Sobol(d=2, scramble=True)
    n_soc = len(soc_levels)
    total_samples = n_soc * num_samples_per_soc
    results = np.empty((total_samples, 3), dtype=float)

    for i, soc in enumerate(soc_levels):
        sampler_log.reset()                               # restart sequence
        sampler_log.fast_forward(i * num_samples_per_soc) #   start of block i
        samples_unit = sampler_log.random(num_samples_per_soc)
        samples_log  = qmc.scale(samples_unit, log_lower, log_upper)

        block = slice(i * num_samples_per_soc, (i + 1) * num_samples_per_soc)
        results[block, 0] = soc
        results[block, 1:] = samples_log                  # (Dan_log, Dca_log)

    return results


In [ ]:
def process_family(...):
    ...
    n_cases = len(diffusivity_params)
    n_train = n_cases * num_train
    n_test  = n_cases * num_test
    n_t, n_r = len(t), len(r)

    train_I              = np.empty((n_train, n_t),           np.float32)
    train_c0_anode       = np.empty((n_train, n_r),           np.float32)
    train_cn_anode       = np.empty((n_train, n_t, n_r),      np.float32)
    train_c0_cathode     = np.empty_like(train_c0_anode)
    train_cn_cathode     = np.empty_like(train_cn_anode)

    test_I               = np.empty((n_test,  n_t),           np.float32)
    test_c0_anode        = np.empty((n_test,  n_r),           np.float32)
    test_cn_anode        = np.empty((n_test,  n_t, n_r),      np.float32)
    test_c0_cathode      = np.empty_like(test_c0_anode)
    test_cn_cathode      = np.empty_like(test_cn_anode)

    # pointer that advances as futures finish
    train_ptr = test_ptr = 0

    with concurrent.futures.ProcessPoolExecutor() as exe:
        futures = [exe.submit(simulate_worker, task) for task in tasks]
        for fut in concurrent.as_completed(futures):
            res = fut.result()
            if res is None:
                continue
            (tr_I, te_I,
             tr_cn_an, tr_c0_an, tr_cn_ca, tr_c0_ca,
             te_cn_an, te_c0_an, te_cn_ca, te_c0_ca) = res

            b_tr, b_te = len(tr_I), len(te_I)
            # copy directly – avoids Python list growth + ragged arrays
            train_I[train_ptr:train_ptr+b_tr]             = tr_I
            train_c0_anode[train_ptr:train_ptr+b_tr]      = tr_c0_an
            train_cn_anode[train_ptr:train_ptr+b_tr]      = tr_cn_an
            train_c0_cathode[train_ptr:train_ptr+b_tr]    = tr_c0_ca
            train_cn_cathode[train_ptr:train_ptr+b_tr]    = tr_cn_ca

            test_I[test_ptr:test_ptr+b_te]                = te_I
            test_c0_anode[test_ptr:test_ptr+b_te]         = te_c0_an
            test_cn_anode[test_ptr:test_ptr+b_te]         = te_cn_an
            test_c0_cathode[test_ptr:test_ptr+b_te]       = te_c0_ca
            test_cn_cathode[test_ptr:test_ptr+b_te]       = te_cn_ca

            train_ptr += b_tr
            test_ptr  += b_te
